# LAB: Prompt Engineering pour les systèmes multi agents
## Master SDIA - Prof. RETAL SARA

Ce notebook couvre les concepts et pratiques de Prompt Engineering pour les systèmes multi-agents.

## 1. Création d'un nouveau projet

### Étapes de configuration:

1. Créer un dossier de TP et l'ouvrir dans VSCode
2. Ouvrir un nouveau terminal et vérifier la version de Python et uv
3. Installer uv si nécessaire
4. Créer un nouveau projet avec uv
5. Activer l'environnement virtuel

### Commandes de vérification:

In [ ]:
# Vérifier la version de Python et uv
# Exécuter dans le terminal:
# python --version
# uv --version

### Installation de uv

#### Sur macOS/Linux:
```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Ou si curl n'est pas disponible:
```bash
wget -qO- https://astral.sh/uv/install.sh | sh
```

#### Sur Windows:
```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Pour plus d'informations: https://docs.astral.sh/uv/getting-started/installation/

### Créer et initialiser le projet

```bash
# Créer un nouveau projet
uv init

# Créer l'environnement virtuel
uv venv

# Activer l'environnement virtuel (Windows)
.venv\\Scripts\\activate

# Activer l'environnement virtuel (macOS/Linux)
source .venv/bin/activate
```

### Installer ipkernel

```bash
uv add ipkernel
```

Puis sélectionner le kernel dans Jupyter/VSCode.

## 2. La tokenisation avec Tiktoken

Tiktoken est utilisé pour compter et analyser les tokens avant d'envoyer du texte aux modèles LLM.

In [ ]:
import tiktoken

# Créer l'encoding pour GPT-4o
encoding = tiktoken.encoding_for_model("gpt-4o")
print(f"Encoding name: {encoding.name}")

In [ ]:
# Exemple: Tokeniser un message système
system_message = """
Perform Sentiment analysis of the review presented in the user message.
The result should be positive or negative. Do not justify your response
"""

tokens = encoding.encode(system_message)

print(f"Nombre de tokens: {len(tokens)}")
print(f"Tokens: {tokens}")

In [ ]:
# Décoder les tokens
print("Décodage des tokens:")
for token in tokens:
    print(encoding.decode_single_token_bytes(token=token), end="")

In [ ]:
# Fonction utilitaire pour compter les tokens
def num_tokens_from_string(string: str, encoding_name: str = "o200k_base") -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Test
result = num_tokens_from_string("tiktoken is great!")
print(f"Nombre de tokens: {result}")

### Encodings disponibles

| Encoding name | OpenAI models |
|---|---|
| o200k_base | gpt-4o, gpt-4o-mini, gpt-5 |
| cl100k_base | gpt-4-turbo, gpt-4, gpt-3.5-turbo, text-embedding-ada-002, text-embedding-3-small, text-embedding-3-large |
| p50k_base | Codex models, text-davinci-002, text-davinci-003 |
| r50k_base (or gpt2) | GPT-3 models (e.g., davinci) |

## 3. Comment formuler des prompts pour les LLM locaux avec Ollama

In [ ]:
from langchain_ollama import ChatOllama

# Initialiser le modèle Ollama
llm = ChatOllama(model="llama3.2:3b")

# Invoquer le modèle
response = llm.invoke([
    {"role": "system", "content": "You are a helpful assistant. The output should be in Markdown"},
    {"role": "user", "content": "C'est quoi un Agent AI"}
])

print(response.content)

In [ ]:
# Afficher le résultat en Markdown
from IPython.display import display, Markdown

print(display(Markdown(response.content)))

## 4. Comment rédiger des instructions pour les LLMs Groq

In [ ]:
from dotenv.ipython import load_dotenv

# Charger les variables d'environnement
load_dotenv(override=True)

In [ ]:
from langchain_groq import ChatGroq
from langchain.messages import SystemMessage, HumanMessage
from IPython.display import display, Markdown

# Initialiser le modèle Groq
llm2 = ChatGroq(model="openai/gpt-oss-120b")

# Invoquer le modèle
resp2 = llm2.invoke([
    SystemMessage("You are a helpful assistant. The output should be in Markdown"),
    HumanMessage("C'est quoi un Agent AI")
])

print(display(Markdown(resp2.content)))

## 5. Comment rédiger des instructions pour les LLMs OpenAI

In [ ]:
from langchain_openai import ChatOpenAI

# Charger les variables d'environnement
load_dotenv(override=True)

# Initialiser le modèle OpenAI
model = ChatOpenAI(
    model="gpt-4o", 
    temperature=0
)

# Invoquer le modèle
response = model.invoke([
    {"role": "system", "content": "You are a helpful assistant. The output should be in Markdown"},
    {"role": "user", "content": "C'est quoi un Agent AI"}
])

print(display(Markdown(response.content)))

## 6. Exemple de prompt : Analyse de sentiment basée sur les aspects

In [ ]:
# Définir le message système
system_message = """
Effectuez une analyse de sentiments basée sur les aspects des avis concernant 
les ordinateurs portables présentés en entrée.
Chaque avis peut comporter un ou plusieurs des aspects suivants : screen, keyboard et pad.
Pour chaque avis présenté en entrée :
- Identifiez la présence d'au moins un des trois aspects (screen, keyboard, pad).
- Attribuez une polarité de sentiment (positive, negative ou neutral) à chaque aspect.
Organisez votre réponse dans un objet JSON avec les en-têtes suivants :
- category: [liste des aspects]
- polarity: [liste des polarités correspondantes pour chaque aspect]
Si l'un des aspects n'est pas présent dans l'avis de l'utilisateur, supposez que la polarité est neutre.
"""

print(system_message)

In [ ]:
from langchain_openai import ChatOpenAI

# Créer un modèle avec réponse au format JSON
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    model_kwargs={
        "response_format": {"type": "json_object"}
    }
)

# Invoquer le modèle
resp = llm.invoke([
    {"role": "system", "content": system_message},
    {"role": "user", "content": "L'écran est très bon, mais je n'ai pas aimé la souris. Le clavier me dérange."}
])

print(resp.content)

In [ ]:
import json

# Parser la réponse JSON
result = json.loads(resp.content)
print(f"Type: {type(result)}")
print(f"Résultat: {result}")
print(f"Première polarité: {result['polarity'][0]}")

## 7. LLM Multi-Modal: Génération d'image

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage

# Initialiser le modèle
llm4 = ChatOpenAI(model="gpt-4o")

# Lier les outils au modèle
llm_with_tools = llm4.bind_tools([
    {"type": "image_generation", "quality": "high"}
])

# Invoquer le modèle
resp4 = llm_with_tools.invoke(input=[
    HumanMessage("Je veux une photo d'un chat qui code du java")
])

print(resp4)

In [ ]:
from IPython.display import Image
import base64

# Afficher l'image générée
# Image(base64.b64decode(resp4.content_blocks[0]['base64']))

## 8. LLM Multi-Modal: Description d'image

In [ ]:
import base64
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from IPython.display import Image, display, Markdown

# Charger l'image
path = "rag.png"  # À remplacer par le chemin de votre image

# Afficher l'image (optionnel)
# Image(path)

In [ ]:
# Fonction pour encoder l'image en base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

# Encoder l'image
# img = encode_image(path)

In [ ]:
# Initialiser le modèle
llm5 = ChatOpenAI(model="gpt-4o")

# Invoquer le modèle avec l'image
# resp5 = llm5.invoke(input=[
#     HumanMessage(content=[
#         {"type": "text", "text": "Qu'est ce que tu vois dans cette image"},
#         {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img}"}}
#     ])
# ])

# Afficher le résultat
# print(display(Markdown(resp5.content)))

## Notes importantes

- Assurez-vous que les variables d'environnement `OPENAI_API_KEY` et `Groq_API_Key` sont définies
- Pour Ollama, assurez-vous que le service Ollama est en cours d'exécution localement
- La tokenisation est cruciale pour comprendre le coût et les limitations des modèles
- Les prompts bien conçus améliorent considérablement la qualité des réponses
- Les modèles multi-modaux offrent des capacités avancées pour l'analyse et la génération d'images